# 🚀 OmniParser Full GPU Server trên Google Colab (100% Original Pipeline)

Notebook này khởi chạy **100% Pipeline gốc của Microsoft OmniParser** (YOLOv8/v9 Icon Detection + Florence-2 Full Semantics Captioning + EasyOCR) trên GPU miễn phí của Google Colab (Tesla T4) và xuất ra đường link công khai (`https://xxxx.trycloudflare.com`) để kết nối về máy tính Linux của bạn.

### Hướng dẫn sử dụng:
1. Trên thanh menu Colab, chọn **Runtime -> Change runtime type -> T4 GPU**.
2. Bấm **Runtime -> Run all** (hoặc chạy lần lượt 2 bước bên dưới).
3. Copy đường link `https://xxxx.trycloudflare.com` xuất hiện ở Bước 2 và dán vào lệnh chạy trên máy Linux của bạn.

In [ ]:
# @title Bước 1: Kiểm tra GPU & Cài đặt thư viện + Tải Model Weights gốc
!nvidia-smi

!git clone https://github.com/doantrongthai/fix_omni_parser.git /content/OmniParser 2>/dev/null || (cd /content/OmniParser && git pull)
%cd /content/OmniParser

# Cài đặt các thư viện cần thiết cho pipeline
!pip install -q ultralytics supervision==0.18.0 easyocr transformers==4.46.3 accelerate timm einops fastapi uvicorn huggingface_hub tabulate groq

# Tải bộ trọng số gốc của Microsoft OmniParser v2.0 (Icon Detect + Florence-2 Caption)
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()
print("⏳ Đang tải mô hình weights từ Hugging Face (~1.4 GB), vui lòng đợi 1-2 phút...")
import os, shutil

os.makedirs("/content/OmniParser/weights", exist_ok=True)
snapshot_download(repo_id="microsoft/OmniParser-v2.0", local_dir="/content/OmniParser/weights")
if os.path.exists("/content/OmniParser/weights/icon_caption") and not os.path.exists("/content/OmniParser/weights/icon_caption_florence"):
    shutil.move("/content/OmniParser/weights/icon_caption", "/content/OmniParser/weights/icon_caption_florence")

# Tải công cụ Cloudflare Tunnel (miễn phí, không cần tài khoản)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("✅ Đã cài đặt xong toàn bộ môi trường và trọng số gốc!")

In [ ]:
# @title Bước 2: Khởi chạy OmniParser GPU Server & Mở Cloudflare Tunnel
import subprocess, threading, re, time, sys, os
from IPython.display import clear_output, display, HTML

# Dọn dẹp tiến trình cũ nếu chạy lại
os.system("pkill -9 -f cloudflared 2>/dev/null")
os.system("pkill -9 -f omniparserserver 2>/dev/null")
time.sleep(1)

public_url_found = False

# 1. Khởi chạy Cloudflare Tunnel
def start_tunnel():
    global public_url_found
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match and not public_url_found:
            public_url_found = True
            public_url = match.group(0)
            with open("/content/public_url.txt", "w") as f:
                f.write(public_url)
            
            # Xóa sạch toàn bộ log rác / progress trước đó và hiển thị nổi bật link công khai
            clear_output(wait=False)
            display(HTML(f"""
            <div style="background-color: #0f172a; color: #f8fafc; padding: 20px; border-radius: 12px; border: 2px solid #38bdf8; font-family: monospace; line-height: 1.6;">
                <h2 style="color: #38bdf8; margin: 0 0 10px 0;">🚀 OMNIPARSER GPU SERVER ĐANG HOẠT ĐỘNG!</h2>
                <p style="font-size: 16px; margin: 5px 0;">🌐 <b>LINK PUBLIC COLAB GPU CỦA BẠN:</b></p>
                <div style="background: #1e293b; padding: 10px 15px; border-radius: 6px; margin: 10px 0; font-size: 18px; color: #4ade80; font-weight: bold; word-break: break-all;">
                    {public_url}
                </div>
                <p style="margin: 10px 0 5px 0;">👉 <b>Lệnh chạy ở Terminal 2 trên máy Linux của bạn:</b></p>
                <pre style="background: #020617; padding: 12px; border-radius: 6px; color: #facc15; font-size: 14px; overflow-x: auto;">python app.py --windows_host_url localhost:5000 --omniparser_server_url {public_url}</pre>
                <p style="color: #94a3b8; font-size: 12px; margin: 5px 0 0 0;">(Link này được lưu tự động tại /content/public_url.txt)</p>
            </div>
            """))

threading.Thread(target=start_tunnel, daemon=True).start()
time.sleep(2)

# 2. Khởi chạy OmniParser FastAPI Server trực tiếp trên GPU CUDA
server_cmd = [
    sys.executable, "-m", "omniparserserver",
    "--device", "cuda",
    "--host", "127.0.0.1",
    "--port", "8000",
    "--BOX_TRESHOLD", "0.05",
    "--use_local_semantics", "true"
]

server_proc = subprocess.Popen(
    server_cmd,
    cwd="/content/OmniParser/omnitool/omniparserserver",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in server_proc.stdout:
    print(line, end="")
